# Analisis exploratorio de datos
### Librerías y unión de bases limpias

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Rutas a los archivos generados en la Etapa 2
carpeta_datos = Path("datos_procesados")

limpios = pd.read_csv(carpeta_datos / "anuncios_limpios.csv",
                      dtype={"id": "string", "host_id": "string"})
codificados = pd.read_csv(carpeta_datos / "anuncios_con_precio_codificados.csv",
                          dtype={"id": "string", "host_id": "string"})

# Unimos el precio por noche a la tabla limpia para el EDA
df = limpios.merge(codificados[["id", "precio_noche_usd"]], on="id", how="left")
df_precio = df.dropna(subset=["precio_noche_usd"]).copy()

print(f"Total anuncios: {len(df):,} | Con precio publicado: {len(df_precio):,}")
display(df_precio.head())

Total anuncios: 13,253 | Con precio publicado: 11,614


,id,host_id,barrio,tipo_habitacion,tipo_propiedad,banos,privacidad_bano,servicios_declarados,declara_wifi,declara_cocina,...,dias_disponibles_365,n_resenas,habitaciones,camas,noches_minimas,habitaciones_imputado,camas_imputado,banos_imputado,noches_minimas_imputado,precio_noche_usd
0,6,29,north hills,entire home/apt,entire home,2.0,sin_especificar,79,1,1,...,248,266,3,6,2,0,0,0,0,316.00
1,12447,48669,pacific beach,private room,private room in condo,1.0,privado,27,1,1,...,342,14,1,1,6,0,1,0,0,146.50
2,62274,302986,pacific beach,private room,private room in home,1.0,privado,34,1,0,...,193,1047,1,1,1,0,0,0,0,121.00
4,723387,3736562,pacific beach,entire home/apt,entire cottage,1.0,sin_especificar,52,1,1,...,309,202,1,1,1,0,0,0,0,282.00
5,741131,1291828,linda vista,entire home/apt,entire loft,1.0,sin_especificar,37,1,1,...,337,59,1,1,31,0,0,0,0,104.98


## Estadísticas descriptivas y verificación de atributos (Univariado)
Calculamos medidas de tendencia central, dispersión, asimetría y curtosis usando las funciones de resumen de Pandas.

In [2]:
num_cols = [
    "precio_noche_usd", "capacidad", "habitaciones", "camas", "banos",
    "servicios_declarados", "noches_minimas", "dias_disponibles_365", "n_resenas"
]

resumen = df_precio[num_cols].describe().T
resumen["asimetria"] = df_precio[num_cols].skew()
resumen["curtosis"] = df_precio[num_cols].kurt()
display(resumen[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "asimetria", "curtosis"]].round(2))

# Calificación únicamente sobre anuncios que sí tienen calificación
calif_validas = df.loc[df["tiene_calificacion"] == 1, "calificacion"]
print("Resumen de calificaciones (anuncios calificados):")
display(calif_validas.describe().to_frame().T.round(2))

,count,mean,std,min,25%,50%,75%,max,asimetria,curtosis
precio_noche_usd,11614.0,379.52,402.91,18.78,162.34,270.38,463.67,9863.0,6.75,99.34
capacidad,11614.0,5.27,3.43,1.00,2.00,4.00,7.00,16.0,1.23,1.27
habitaciones,11614.0,1.99,1.44,0.00,1.00,2.00,3.00,32.0,2.21,20.47
camas,11614.0,2.93,2.41,1.00,1.00,2.00,4.00,37.0,3.48,26.43
banos,11614.0,1.68,1.05,0.00,1.00,1.00,2.00,23.0,3.06,25.58
servicios_declarados,11614.0,45.92,16.99,0.00,34.00,47.00,58.00,110.0,-0.18,-0.33
noches_minimas,11614.0,9.88,16.14,1.00,2.00,2.00,30.00,365.0,6.68,113.84
dias_disponibles_365,11614.0,235.25,109.17,0.00,150.00,264.00,332.00,365.0,-0.59,-0.94
n_resenas,11614.0,79.08,124.73,0.00,3.00,30.00,101.00,1918.0,3.39,20.12


Resumen de calificaciones (anuncios calificados):


,count,mean,std,min,25%,50%,75%,max
calificacion,11099.0,4.81,0.33,1.0,4.77,4.9,4.98,5.0


Esta tabla resume el comportamiento numérico de los 11.614 anuncios con precio publicado y los 11.099 con calificación, evidenciando la presencia de valores extremos y asimetrías